# 4.5 一層模型怎麼組成？

# 第 4 章：第一個 Dense Transformer 語言模型

前置：第3章的因果 attention。完整可讀實作在 tiny_perceptron/model.py；預設只用 learned position、LayerNorm、Dense FFN，還沒有 FlashAttention 或 MoE。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像原稿旁加修訂，而不是撕掉原稿**

y=x+f(x)，原資料和修訂都有路徑。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/residual.svg")))

**先想一想：**block 會把序列長度或hidden size改變嗎？

先把規範化、attention、residual接起來，再接規範化、FFN、residual。pre-norm 是先正規化才進分支，主路徑仍保留原表示。

**把直覺寫成數學：**$u=x+Attention(LN(x))$；$y=u+FFN(LN(u))$。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import Block, ModelConfig

block = Block(ModelConfig(width=8))
x = torch.randn(2, 3, 8)
y, cache, auxiliary = block(x)
print(x.shape, y.shape, auxiliary.item())
assert y.shape == x.shape

**如何讀結果：**blocks可堆疊，因為輸入輸出契約相同；auxiliary在Dense基準是0。

**只改一件事：**只把batch數改成1。
